# ❗ 5. SKHynix PBL 시계열 시퀀스 모델링 2❗


## 📌 개요

시간대(timekey_hr) 내에서 공정 순서(oper_id)를 고려한 시퀀스 기반 TAT 예측 모델입니다. 동일한 timekey_hr 내의 oper_id들을 순서대로 정렬하여 시퀀스 데이터로 구성하고, 각 oper별 개별 예측(sequence-to-sequence)을 수행합니다.

**데이터 구조**: `[batch_size, sequence_length, feature_dim]`
- **sequence_length**: timekey_hr 내 함께 구성될 oper_id 개수 + 가장 많은 group의 oper_id 수
- **feature_dim**: 연속형 변수 개수 + 범주형 변수 개수 × 임베딩 차원

## 🔧 환경 설정 및 라이브러리

In [11]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import yaml
import logging
import json
import argparse
import math

from datetime import datetime
from tqdm import tqdm
from typing import Dict, List, Tuple, Optional, Union
from types import SimpleNamespace
from collections import defaultdict

# sklearn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# PyTorch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR

# Oper Transformer

## 📊 유틸리티 함수들

### 설정 로딩 및 시드 설정

In [12]:
def load_config(config_dir: str = "configs") -> Dict:
    """YAML 설정 파일들을 통합하여 로드"""
    configs = {}
    config_files = ["dataset", "model", "training"]

    for file in config_files:
        config_path = os.path.join(config_dir, f"{file}.yaml")
        with open(config_path, "r", encoding="utf-8") as f:
            config = yaml.safe_load(f)
            configs.update(config)

    return configs


def set_random_seeds(seed: int = 42):
    """재현성을 위한 랜덤 시드 설정"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def setup_logging(log_file: str = "training.log"):
    """로깅 설정"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

logger = setup_logging()

### 범주형 데이터 처리기

In [13]:
class CategoricalProcessor:
    """범주형 변수 임베딩을 위한 처리기"""
    
    def __init__(self, embedding_dim: int = 8):
        self.embedding_dim = embedding_dim
        self.label_encoders = {}
        self.vocab_sizes = {}
        self.categorical_columns = []
        
    def fit(self, df: pd.DataFrame, categorical_columns: List[str]):
        """전체 데이터에 대해 범주형 인코더 학습"""
        self.categorical_columns = categorical_columns
        
        for col in categorical_columns:
            unique_values = df[col].astype(str).unique()
            encoder = LabelEncoder()
            encoder.fit(unique_values)
            
            self.label_encoders[col] = encoder
            self.vocab_sizes[col] = len(encoder.classes_)
        
        logger.info(f"범주형 변수별 고유값 개수:")
        for col in categorical_columns:
            logger.info(f"  {col}: {self.vocab_sizes[col]}개")
    
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """DataFrame의 범주형 컬럼들을 숫자로 변환"""
        df_encoded = df.copy()
        
        for col in self.categorical_columns:
            df_encoded[col] = self.label_encoders[col].transform(
                df_encoded[col].astype(str)
            )
        
        return df_encoded
    
    def get_vocab_sizes(self) -> List[int]:
        """각 범주형 변수의 vocab_size 리스트 반환"""
        return [self.vocab_sizes[col] for col in self.categorical_columns]

## 🗂️ 시퀀스 데이터셋 클래스

### 메인 데이터셋

In [14]:
class GroupedOperDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        categorical_columns: List[str],
        continuous_columns: List[str],
        target_column: str = "y",
        group_map: Dict = None,  # {group_id: [oper_ids]}
        window_size: int = 10,
        window_tride: int = 1,
        padding_value: float = 0.0
    ):
        self.df = df.copy()
        self.categorical_columns = categorical_columns
        self.continuous_columns = continuous_columns
        self.target_column = target_column
        self.group_map = group_map
        self.window_size = window_size
        self.window_tride = window_tride
        self.padding_value = padding_value
        
        # 특성 차원 계산
        self.continuous_dim = len(continuous_columns)
        self.categorical_dim = len(categorical_columns)
        self.feature_dim = self.continuous_dim + self.categorical_dim
        
        # timekey_hr 기준 정렬
        self.df = self.df.sort_values(['timekey_hr', 'oper_id']).reset_index(drop=True)
        
        # 최대 그룹 크기 계산
        self.max_group_size = max(len(opers) for opers in self.group_map.values())
        
        # 전체 시퀀스 길이 = window_size + max_group_size
        self.total_sequence_length = self.window_size + self.max_group_size
        
        # 시퀀스 생성
        self._create_sequences()
        
        logger.info(f"Window-Group 데이터셋 구성 완료:")
        logger.info(f"  - 총 시퀀스 수: {len(self.sequences)}")
        logger.info(f"  - Window 크기: {window_size}")
        logger.info(f"  - Stride: {window_tride}")
        logger.info(f"  - 최대 그룹 크기: {self.max_group_size}")
        logger.info(f"  - 전체 시퀀스 길이: {self.total_sequence_length}")
        logger.info(f"  - 특성 차원: {self.feature_dim}")
        logger.info(f"  - 패딩값: {padding_value}")
    
    def _create_sequences(self):
        """Window sliding 기반 시퀀스 생성"""
        self.sequences = []
        
        # 고유 timekey_hr 목록
        unique_timekeys = sorted(self.df['timekey_hr'].unique())
        
        # Window sliding
        for start_idx in range(0, len(unique_timekeys) - self.window_size + 1, self.window_tride):
            window_timekeys = unique_timekeys[start_idx:start_idx + self.window_size]
            first_timekey = window_timekeys[0]
            
            # 첫 번째 시간대의 각 그룹별로 시퀀스 생성
            first_time_data = self.df[self.df['timekey_hr'] == first_timekey]
            unique_groups = first_time_data['oper_group'].unique()
            
            for target_group in unique_groups:
                # Window 데이터 수집 (각 시간대의 그룹 평균/대표값)
                window_continuous = []
                window_categorical = []
                window_targets = []
                
                for timekey in window_timekeys:
                    time_group_data = self.df[
                        (self.df['timekey_hr'] == timekey) & 
                        (self.df['oper_group'] == target_group)
                    ]
                    
                    if len(time_group_data) > 0:
                        # 그룹의 대표값 (평균 또는 첫 번째 작업자)
                        cont_values = time_group_data[self.continuous_columns].mean().values
                        cat_values = time_group_data[self.categorical_columns].iloc[0].values
                        target_value = time_group_data[self.target_column].mean()
                    else:
                        # 해당 시간대에 그룹 데이터 없음
                        cont_values = np.full(self.continuous_dim, self.padding_value)
                        cat_values = np.zeros(self.categorical_dim)
                        target_value = self.padding_value
                    
                    window_continuous.append(cont_values)
                    window_categorical.append(cat_values)
                    window_targets.append(target_value)
                
                # Group 데이터 수집 (첫 시간대, 해당 그룹의 모든 작업자)
                group_data = self.df[
                    (self.df['timekey_hr'] == first_timekey) & 
                    (self.df['oper_group'] == target_group)
                ].sort_values('oper_id')
                
                if len(group_data) == 0:
                    continue
                
                group_continuous = group_data[self.continuous_columns].values
                group_categorical = group_data[self.categorical_columns].values
                group_targets = group_data[self.target_column].values
                group_oper_ids = group_data['oper_id'].values
                
                sequence_info = {
                    'window_continuous': np.array(window_continuous, dtype=np.float32),
                    'window_categorical': np.array(window_categorical, dtype=np.float32),
                    'window_targets': np.array(window_targets, dtype=np.float32),
                    'group_continuous': group_continuous.astype(np.float32),
                    'group_categorical': group_categorical.astype(np.float32),
                    'group_targets': group_targets.astype(np.float32),
                    'group_oper_ids': group_oper_ids,
                    'group_size': len(group_data),
                    'target_group': target_group,
                    'first_timekey': first_timekey,
                    'window_timekeys': window_timekeys
                }
                
                self.sequences.append(sequence_info)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        sequence = self.sequences[idx]
        
        # 패딩된 배열 초기화
        continuous_data = np.full(
            (self.total_sequence_length, self.continuous_dim),
            self.padding_value, dtype=np.float32
        )
        categorical_data = np.full(
            (self.total_sequence_length, self.categorical_dim),
            0, dtype=np.float32  # 범주형은 0으로 패딩
        )
        target_data = np.full(
            self.total_sequence_length,
            self.padding_value, dtype=np.float32
        )
        
        # Window 데이터 채우기 (0 ~ window_size-1)
        continuous_data[:self.window_size] = sequence['window_continuous']
        categorical_data[:self.window_size] = sequence['window_categorical']
        target_data[:self.window_size] = sequence['window_targets']
        
        # Group 데이터 채우기 (window_size ~ window_size+group_size-1)
        group_size = min(sequence['group_size'], self.max_group_size)
        if group_size > 0:
            group_start = self.window_size
            group_end = group_start + group_size
            continuous_data[group_start:group_end] = sequence['group_continuous'][:group_size]
            categorical_data[group_start:group_end] = sequence['group_categorical'][:group_size]
            target_data[group_start:group_end] = sequence['group_targets'][:group_size]
        
        # 마스크 생성 (True = 실제 데이터, False = 패딩)
        mask = np.zeros(self.total_sequence_length, dtype=bool)
        mask[:self.window_size] = True  # Window 부분
        if group_size > 0:
            mask[self.window_size:self.window_size + group_size] = True  # Group 부분
        
        # Position indicator (0: window, 1: group, -1: padding)
        position_ids = np.full(self.total_sequence_length, -1, dtype=np.int32)
        position_ids[:self.window_size] = 0  # Window
        if group_size > 0:
            position_ids[self.window_size:self.window_size + group_size] = 1  # Group
        
        # oper_ids 리스트 생성 (구조 정보용)
        oper_ids_list = [None] * self.total_sequence_length
        # Window 부분은 그룹 대표값이므로 그룹 ID를 사용
        for i in range(self.window_size):
            oper_ids_list[i] = f"group_{sequence['target_group']}"
        # Group 부분은 실제 oper_id 사용
        if group_size > 0:
            for i in range(group_size):
                oper_ids_list[self.window_size + i] = sequence['group_oper_ids'][i]
            
        return {
            'continuous_data': torch.tensor(continuous_data, dtype=torch.float32),
            'categorical_data': torch.tensor(categorical_data, dtype=torch.float32),
            'targets': torch.tensor(target_data, dtype=torch.float32),
            'masks': torch.tensor(mask, dtype=torch.bool),
            'position_ids': torch.tensor(position_ids, dtype=torch.long),
            'sequence_lengths': self.total_sequence_length,
            # 구조 정보 추가
            'timekey_hrs': sequence['first_timekey'],  # 시퀀스의 첫 timekey
            'oper_ids_list': oper_ids_list,  # oper_id 리스트
            'window_timekeys': sequence['window_timekeys'],  # 전체 window의 timekey 리스트
            'target_group': sequence['target_group']  # 타겟 그룹
        }
        
def custom_collate_fn(batch):
    """커스텀 배치 처리 함수"""
    # 텐서로 변환 가능한 필드들
    continuous_data = torch.stack([item['continuous_data'] for item in batch])
    categorical_data = torch.stack([item['categorical_data'] for item in batch])
    targets = torch.stack([item['targets'] for item in batch])
    masks = torch.stack([item['masks'] for item in batch])
    position_ids = torch.stack([item['position_ids'] for item in batch])
    
    # 리스트나 스칼라 값들은 리스트로 유지
    sequence_lengths = [item['sequence_lengths'] for item in batch]
    timekey_hrs = [item['timekey_hrs'] for item in batch]
    oper_ids_list = [item['oper_ids_list'] for item in batch]
    window_timekeys = [item['window_timekeys'] for item in batch]
    target_groups = [item['target_group'] for item in batch]
    
    return {
        'continuous_data': continuous_data,
        'categorical_data': categorical_data,
        'targets': targets,
        'masks': masks,
        'position_ids': position_ids,
        'sequence_lengths': sequence_lengths,
        'timekey_hrs': timekey_hrs,
        'oper_ids_list': oper_ids_list,
        'window_timekeys': window_timekeys,
        'target_group': target_groups
    }


In [ ]:
def create_dataloaders(config: Dict) -> Tuple[DataLoader, DataLoader, DataLoader]:

    # 데이터 로드 및 전처리
    data_path = config["file_path"]
    # nrows는 테스트용이므로 꼭 제거 !
    excel = pd.read_excel(data_path, sheet_name=None, header=1)
    sheet_names = config["sheet_names"]

    total_df = pd.concat([excel[sheet_name] for sheet_name in sheet_names])
    group_to_opers = total_df.groupby('oper_group')['oper_id'].unique().to_dict()

    # 기본 전처리
    if "Unnamed: 0" in total_df.columns:
        total_df.drop(columns="Unnamed: 0", inplace=True)

    # y값 결측치 제거
    df = total_df[~total_df[config["target_column"]].isna()].copy()

    # 불필요한 컬럼 제거
    drop_columns = config.get("additional_drop_columns", [])
    if drop_columns:
        existing_drops = [col for col in drop_columns if col in df.columns]
        if existing_drops:
            df = df.drop(columns=existing_drops)

    df.reset_index(drop=True, inplace=True)

    # 전체 데이터에 대해 범주형 처리기 학습
    categorical_processor = CategoricalProcessor(
        embedding_dim=config.get("embedding_dim", 8)
    )
    categorical_processor.fit(df, config["categorical_columns"])
    df = categorical_processor.transform(df)

    # 데이터 분할 (8:1:1)
    total_size = len(df)
    train_end = int(total_size * config.get("train_ratio", 0.8))
    val_end = int(total_size * (config.get("train_ratio", 0.8) + config.get("val_ratio", 0.1)))

    train_df = df[:train_end].copy()
    val_df = df[train_end:val_end].copy()
    test_df = df[val_end:].copy()
            
    # 데이터셋 생성
    train_dataset = GroupedOperDataset(
        df=train_df,
        categorical_columns=config["categorical_columns"],
        continuous_columns=config["continuous_columns"],
        target_column=config["target_column"],
        group_map=group_to_opers,
        window_size=config.get("window_size", 10),
        window_tride=config.get("window_tride", 1),
        padding_value=config.get("padding_value", 0.0)
    )

    val_dataset = GroupedOperDataset(
        df=val_df,
        categorical_columns=config["categorical_columns"],
        continuous_columns=config["continuous_columns"], 
        target_column=config["target_column"],
        group_map=group_to_opers,
        window_size=config.get("window_size", 10),
        window_tride=config.get("window_tride", 1),
        padding_value=config.get("padding_value", 0.0)
    )

    test_dataset = GroupedOperDataset(
        df=test_df,
        categorical_columns=config["categorical_columns"],
        continuous_columns=config["continuous_columns"],
        target_column=config["target_column"],
        group_map=group_to_opers,
        window_size=config.get("window_size", 10),
        window_tride=config.get("window_tride", 1),
        padding_value=config.get("padding_value", 0.0)
    )

    # 데이터로더 생성
    batch_size = config.get("batch_size", 32)
    num_workers = config.get("num_workers", 4)

    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=custom_collate_fn
    )

    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=custom_collate_fn
    )

    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=custom_collate_fn
    )
    

    logger.info(f"데이터로더 생성 완료:")
    logger.info(f"  - 훈련 샘플: {len(train_dataset)}")
    logger.info(f"  - 검증 샘플: {len(val_dataset)}")
    logger.info(f"  - 테스트 샘플: {len(test_dataset)}")
    logger.info(f"  - 배치 크기: {batch_size}")

    return train_loader, val_loader, test_loader, categorical_processor

## OperTransformer모델

In [16]:
"""Encoding Modules"""

class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEmbedding, self).__init__()
        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model).float()
        pe.require_grad = False

        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = (torch.arange(0, d_model, 2).float()
                    * -(math.log(10000.0) / d_model)).exp()

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return self.pe[:, :x.size(1)]


class CategoricalEmbedding(nn.Module):
    """범주형 변수를 위한 임베딩 모듈"""
    def __init__(self, vocab_sizes: List[int], embedding_dim: int):
        super(CategoricalEmbedding, self).__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_size, embedding_dim) 
            for vocab_size in vocab_sizes
        ])
        
    def forward(self, categorical_inputs):
        """
        Args:
            categorical_inputs: [batch_size, seq_len, num_categorical_features]
        Returns:
            embedded: [batch_size, seq_len, num_categorical_features * embedding_dim]
        """
        embedded_features = []
        for i, embedding_layer in enumerate(self.embeddings):
            # 각 범주형 변수에 대해 임베딩 수행
            cat_input = categorical_inputs[:, :, i].long()
            embedded = embedding_layer(cat_input)
            embedded_features.append(embedded)
        
        # 모든 임베딩을 concatenate
        return torch.cat(embedded_features, dim=-1)


class DataEmbeddingWithCategorical(nn.Module):
    """범주형 및 연속형 변수를 모두 처리하는 임베딩 모듈"""
    def __init__(self, 
                 continuous_dim: int,
                 vocab_sizes: List[int],
                 d_model: int,
                 window_size: int = 10,
                 embedding_dim: int = 8,
                 dropout: float = 0.1):
        super(DataEmbeddingWithCategorical, self).__init__()
        
        # window_size 저장
        self.window_size = window_size
        
        # 범주형 변수 임베딩
        self.categorical_embedding = CategoricalEmbedding(vocab_sizes, embedding_dim)
        
        # 범주형 임베딩과 연속형 변수를 결합한 차원
        total_input_dim = continuous_dim + len(vocab_sizes) * embedding_dim
        
        # 입력을 d_model 차원으로 투영
        self.input_projection = nn.Linear(total_input_dim, d_model)
        
        # Positional 임베딩
        self.position_embedding = PositionalEmbedding(d_model)
        
        self.dropout = nn.Dropout(p=dropout)
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, continuous_x, categorical_x):
        """
        Args:
            continuous_x: [batch_size, seq_len, continuous_dim]
            categorical_x: [batch_size, seq_len, num_categorical_features]
        Returns:
            embedded: [batch_size, seq_len, d_model]
        """
        batch_size, seq_len = continuous_x.shape[:2]
        
        # 범주형 변수 임베딩
        cat_embedded = self.categorical_embedding(categorical_x)
        
        # 연속형과 범주형 결합
        combined = torch.cat([continuous_x, cat_embedded], dim=-1)
        
        # d_model 차원으로 투영
        x = self.input_projection(combined)
        
        # Positional 임베딩을 window_size까지만 추가
        if seq_len <= self.window_size:
            # 시퀀스가 window_size보다 작거나 같으면 전체에 적용
            x = x + self.position_embedding(x)
        else:
            # window_size까지만 positional encoding 적용
            pos_encoding = self.position_embedding(x[:, :self.window_size, :])
            x[:, :self.window_size, :] = x[:, :self.window_size, :] + pos_encoding
            # window_size 이후는 positional encoding 없이 그대로 유지
        
        # Normalization과 Dropout
        x = self.norm(x)
        x = self.dropout(x)
        
        return x

"""Encoder/Decoder Modules"""

class ConvLayer(nn.Module):
    def __init__(self, c_in):
        super(ConvLayer, self).__init__()
        self.downConv = nn.Conv1d(in_channels=c_in,
                                  out_channels=c_in,
                                  kernel_size=3,
                                  padding=2,
                                  padding_mode='circular')
        self.norm = nn.BatchNorm1d(c_in)
        self.activation = nn.ELU()
        self.maxPool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        x = self.downConv(x.permute(0, 2, 1))
        x = self.norm(x)
        x = self.activation(x)
        x = self.maxPool(x)
        x = x.transpose(1, 2)
        return x


class EncoderLayer(nn.Module):
    def __init__(self, attention, d_model, d_ff=None, dropout=0.1, activation="relu"):
        super(EncoderLayer, self).__init__()
        d_ff = d_ff or 4 * d_model
        self.attention = attention
        self.conv1 = nn.Conv1d(in_channels=d_model, out_channels=d_ff, kernel_size=1)
        self.conv2 = nn.Conv1d(in_channels=d_ff, out_channels=d_model, kernel_size=1)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation == "relu" else F.gelu

    def forward(self, x, attn_mask=None, tau=None, delta=None):
        new_x, attn = self.attention(
            x, x, x,
            attn_mask=attn_mask,
            tau=tau, delta=delta
        )
        x = x + self.dropout(new_x)

        y = x = self.norm1(x)
        y = self.dropout(self.activation(self.conv1(y.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))

        return self.norm2(x + y), attn


class Encoder(nn.Module):
    def __init__(self, attn_layers, conv_layers=None, norm_layer=None):
        super(Encoder, self).__init__()
        self.attn_layers = nn.ModuleList(attn_layers)
        self.conv_layers = nn.ModuleList(conv_layers) if conv_layers is not None else None
        self.norm = norm_layer

    def forward(self, x, attn_mask=None, tau=None, delta=None):
        # x [B, L, D]
        attns = []
        if self.conv_layers is not None:
            for i, (attn_layer, conv_layer) in enumerate(zip(self.attn_layers, self.conv_layers)):
                delta = delta if i == 0 else None
                x, attn = attn_layer(x, attn_mask=attn_mask, tau=tau, delta=delta)
                x = conv_layer(x)
                attns.append(attn)
            x, attn = self.attn_layers[-1](x, tau=tau, delta=None)
            attns.append(attn)
        else:
            for attn_layer in self.attn_layers:
                x, attn = attn_layer(x, attn_mask=attn_mask, tau=tau, delta=delta)
                attns.append(attn)

        if self.norm is not None:
            x = self.norm(x)

        return x, attns


class DecoderLayer(nn.Module):
    def __init__(self, self_attention, cross_attention, d_model, d_ff=None,
                 dropout=0.1, activation="relu"):
        super(DecoderLayer, self).__init__()
        d_ff = d_ff or 4 * d_model
        self.self_attention = self_attention
        self.cross_attention = cross_attention
        self.conv1 = nn.Conv1d(in_channels=d_model, out_channels=d_ff, kernel_size=1)
        self.conv2 = nn.Conv1d(in_channels=d_ff, out_channels=d_model, kernel_size=1)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation == "relu" else F.gelu

    def forward(self, x, cross, x_mask=None, cross_mask=None, tau=None, delta=None):
        x = x + self.dropout(self.self_attention(
            x, x, x,
            attn_mask=x_mask,
            tau=tau, delta=None
        )[0])
        x = self.norm1(x)

        x = x + self.dropout(self.cross_attention(
            x, cross, cross,
            attn_mask=cross_mask,
            tau=tau, delta=delta
        )[0])

        y = x = self.norm2(x)
        y = self.dropout(self.activation(self.conv1(y.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))

        return self.norm3(x + y)


class Decoder(nn.Module):
    def __init__(self, layers, norm_layer=None, projection=None):
        super(Decoder, self).__init__()
        self.layers = nn.ModuleList(layers)
        self.norm = norm_layer
        self.projection = projection

    def forward(self, x, cross, x_mask=None, cross_mask=None, tau=None, delta=None):
        for layer in self.layers:
            x = layer(x, cross, x_mask=x_mask, cross_mask=cross_mask, tau=tau, delta=delta)

        if self.norm is not None:
            x = self.norm(x)

        if self.projection is not None:
            x = self.projection(x)
        return x

"""Attention Modules"""

class AttentionLayer(nn.Module):
    def __init__(self, attention, d_model, n_heads, d_keys=None,
                 d_values=None):
        super(AttentionLayer, self).__init__()

        d_keys = d_keys or (d_model // n_heads)
        d_values = d_values or (d_model // n_heads)

        self.inner_attention = attention
        self.query_projection = nn.Linear(d_model, d_keys * n_heads)
        self.key_projection = nn.Linear(d_model, d_keys * n_heads)
        self.value_projection = nn.Linear(d_model, d_values * n_heads)
        self.out_projection = nn.Linear(d_values * n_heads, d_model)
        self.n_heads = n_heads

    def forward(self, queries, keys, values, attn_mask, tau=None, delta=None):
        B, L, _ = queries.shape
        _, S, _ = keys.shape
        H = self.n_heads

        queries = self.query_projection(queries).view(B, L, H, -1)
        keys = self.key_projection(keys).view(B, S, H, -1)
        values = self.value_projection(values).view(B, S, H, -1)

        out, attn = self.inner_attention(
            queries,
            keys,
            values,
            attn_mask,
            tau=tau,
            delta=delta
        )
        out = out.view(B, L, -1)

        return self.out_projection(out), attn

class TriangularCausalMask():
    def __init__(self, B, L, device="cpu"):
        mask_shape = [B, 1, L, L]
        with torch.no_grad():
            self._mask = torch.triu(torch.ones(mask_shape, dtype=torch.bool), diagonal=1).to(device)

    @property
    def mask(self):
        return self._mask
    
class FullAttention(nn.Module):
    def __init__(self, mask_flag=True, factor=5, scale=None, attention_dropout=0.1, output_attention=False):
        super(FullAttention, self).__init__()
        self.scale = scale
        self.mask_flag = mask_flag
        self.output_attention = output_attention
        self.dropout = nn.Dropout(attention_dropout)

    def forward(self, queries, keys, values, attn_mask, tau=None, delta=None):
        B, L, H, E = queries.shape
        _, S, _, D = values.shape
        scale = self.scale or 1. / math.sqrt(E)

        scores = torch.einsum("blhe,bshe->bhls", queries, keys)

        if self.mask_flag:
            if attn_mask is None:
                attn_mask = TriangularCausalMask(B, L, device=queries.device)

            scores.masked_fill_(attn_mask.mask, -np.inf)

        A = self.dropout(torch.softmax(scale * scores, dim=-1))
        V = torch.einsum("bhls,bshd->blhd", A, values)

        if self.output_attention:
            return V.contiguous(), A
        else:
            return V.contiguous(), None


"""Model"""

class VanillaTransformer(nn.Module):
    """사용자 데이터 구조에 맞춘 수정된 Vanilla Transformer"""
    
    def __init__(self, configs):
        super(VanillaTransformer, self).__init__()
        
        
        # 데이터 임베딩 (범주형 + 연속형)
        self.data_embedding = DataEmbeddingWithCategorical(
            continuous_dim=configs.continuous_dim,
            vocab_sizes=configs.vocab_sizes,
            d_model=configs.d_model,
            embedding_dim=configs.embedding_dim,
            dropout=configs.dropout,
            window_size=configs.window_size 
        )
        
        # Encoder
        self.encoder = Encoder(
            [
                EncoderLayer(
                    AttentionLayer(
                        FullAttention(
                            mask_flag=False,
                            factor=configs.factor,
                            attention_dropout=configs.dropout,
                            output_attention=False
                        ),
                        configs.d_model,
                        configs.n_heads
                    ),
                    configs.d_model,
                    configs.d_ff,
                    dropout=configs.dropout,
                    activation=configs.activation
                ) for _ in range(configs.e_layers)
            ],
            norm_layer=nn.LayerNorm(configs.d_model)
        )
        
        # Prediction head
        self.prediction_head = nn.Sequential(
            nn.Linear(configs.d_model, configs.d_ff),
            nn.ReLU(),
            nn.Dropout(configs.dropout),
            nn.Linear(configs.d_ff, 1)  # 단일 값 예측
        )
        
        self.output_type = configs.output_type  # 'sequence' or 'last'
        
    def forward(self, continuous_data, categorical_data, masks=None, sequence_lengths=None):
        """
        학습 코드와 호환되는 forward 메서드
        
        Args:
            continuous_data: [batch_size, seq_len, continuous_dim]
            categorical_data: [batch_size, seq_len, num_categorical]
            masks: [batch_size, seq_len] - True는 패딩 위치
            sequence_lengths: 실제 시퀀스 길이 (선택적)
            
        Returns:
            predictions: [batch_size, seq_len] or [batch_size]
        """
        # 임베딩
        
        enc_out = self.data_embedding(continuous_data, categorical_data)
        
        # Attention mask 처리
        attn_mask = None
        if masks is not None:
            # masks를 attention mask로 변환 (True = 마스킹할 위치)
            batch_size, seq_len = masks.shape
            attn_mask = masks.unsqueeze(1).unsqueeze(1)  # [B, 1, 1, L]
            attn_mask = attn_mask.expand(-1, 1, seq_len, -1)  # [B, 1, L, L]
        
        # Encoder
        enc_out, attns = self.encoder(enc_out, attn_mask=attn_mask)
        
        if self.output_type == 'sequence':
            # 모든 타임스텝에 대해 예측
            predictions = self.prediction_head(enc_out).squeeze(-1)  # [B, L]
        else:
            # 마지막 유효한 타임스텝만 사용
            if sequence_lengths is not None:
                # 각 샘플의 마지막 유효 위치 추출
                batch_size = enc_out.shape[0]
                last_hidden = []
                for i in range(batch_size):
                    last_idx = min(sequence_lengths[i] - 1, enc_out.shape[1] - 1)
                    last_hidden.append(enc_out[i, last_idx])
                last_hidden = torch.stack(last_hidden)
            else:
                # 단순히 마지막 위치 사용
                last_hidden = enc_out[:, -1, :]
            
            predictions = self.prediction_head(last_hidden).squeeze(-1)  # [B]
            
            # 시퀀스 형태로 확장 (학습 코드 호환성)
            if masks is not None:
                batch_size, seq_len = masks.shape
                predictions_expanded = predictions.unsqueeze(1).expand(-1, seq_len)
                # 마지막 유효 위치에만 예측값 할당
                predictions = torch.zeros_like(masks, dtype=torch.float32)
                for i in range(batch_size):
                    if sequence_lengths is not None:
                        last_idx = min(sequence_lengths[i] - 1, seq_len - 1)
                    else:
                        last_idx = seq_len - 1
                    predictions[i, last_idx] = predictions_expanded[i, 0]
            else:
                predictions = predictions.unsqueeze(1)  # [B, 1]
        
        return predictions


### 실험 설정 (모델 생성/Loss 설정)

In [17]:
def create_model(config: Dict, vocab_sizes: List[int], continuous_dim: int):
    """설정에 따른 모델 생성"""
    
    model_config = {
        # 데이터 차원
        "continuous_dim": continuous_dim,
        "vocab_sizes": vocab_sizes,
        "output_type": 'sequence',
        
        # 모델 하이퍼파라미터
        "seq_len": config.get("seq_len", 100),
        "pred_len": config.get("pred_len", 1),
        "window_size": config.get("window_size", 10),
        
        # Transformer 설정
        "d_model": config.get("d_model", 512),
        "n_heads": config.get("n_heads", 8),
        "e_layers": config.get("e_layers", 3),
        "d_ff": config.get("d_ff", 2048),
        
        # 임베딩 설정
        "embedding_dim": config.get("embedding_dim", 8),
        
        # 학습 설정
        "dropout": config.get("dropout", 0.1),
        "activation": config.get("activation", "relu"),
        "factor": config.get("factor", 5),
        
        # 태스크 설정
        "task_name": config.get("task_name", "regression"),
    }
    
    model_config = SimpleNamespace(**model_config)
    
    model = VanillaTransformer(model_config)
    
    logger.info(f"모델 생성 완료:")
    logger.info(f"  - 총 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
    logger.info(f"  - 학습 가능한 파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    return model

class MSELoss(nn.Module):    
    def __init__(self, padding_value: float = 0.0):
        super().__init__()
        self.padding_value = padding_value
    
    def forward(self, predictions, targets, masks):
        """
        Args:
            predictions: [batch_size, seq_len]
            targets: [batch_size, seq_len]  
            masks: [batch_size, seq_len] (True = 패딩)
        """
        # # 패딩되지 않은 위치만 선택
        # valid_mask = ~masks
        
        # if valid_mask.sum() == 0:
        #     return torch.tensor(0.0, device=predictions.device, requires_grad=True)
        
        # valid_predictions = predictions[valid_mask]
        # valid_targets = targets[valid_mask]
        
        return F.mse_loss(predictions, targets)


def compute_metrics(predictions, targets, masks, padding_value: float = 0.0, exclude_zeros=True):
    """
    패딩을 고려한 메트릭 계산
    
    Args:
        predictions: 예측값 텐서
        targets: 실제값 텐서  
        masks: 패딩 마스크 (True가 패딩)
        padding_value: 패딩 값
        exclude_zeros: True면 MAPE/SMAPE 계산 시 타겟 0 제외,
                      False면 0도 포함 (epsilon으로 처리)
    """
    
    # CPU로 변환 후 numpy 배열로 변환
    valid_predictions = predictions.detach().cpu().numpy()
    valid_targets = targets.detach().cpu().numpy()
    
    # 기본 메트릭 (모든 유효 데이터 포함)
    mse = np.mean((valid_predictions - valid_targets) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(valid_predictions - valid_targets))
    
    if exclude_zeros:
        # 타겟이 0이 아닌 경우만 필터링
        non_zero_mask = valid_targets != 0
        non_zero_count = non_zero_mask.sum()
        
        if non_zero_count > 0:
            filtered_predictions = valid_predictions[non_zero_mask]
            filtered_targets = valid_targets[non_zero_mask]
            
            abs_errors = np.abs(filtered_predictions - filtered_targets)
            abs_targets = np.abs(filtered_targets)
            
            # MAPE
            mape = np.mean(abs_errors / abs_targets * 100)
            
            # SMAPE
            smape = 100 * np.mean(
                2 * abs_errors / (abs_targets + np.abs(filtered_predictions) + 1e-8)
            )
        else:
            mape = np.nan
            smape = np.nan
            
    else:
        # 0도 포함하여 계산 (epsilon 사용)
        epsilon = 1e-8
        abs_targets = np.abs(valid_targets)
        abs_errors = np.abs(valid_predictions - valid_targets)
        
        # MAPE - epsilon으로 0 처리
        safe_targets = np.maximum(abs_targets, epsilon)
        mape = np.mean(abs_errors / safe_targets * 100)
        
        # SMAPE - epsilon으로 0 처리
        denominator = (abs_targets + np.abs(valid_predictions)) / 2 + epsilon
        smape = 100 * np.mean(abs_errors / denominator)
        
        non_zero_count = (valid_targets != 0).sum()
    
    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "mape": mape,
        "smape": smape,
        "valid_count": len(valid_predictions),
        "non_zero_count": non_zero_count,
        "zero_ratio": (len(valid_targets) - non_zero_count) / len(valid_targets) * 100,
        "exclude_zeros": exclude_zeros  # 어떤 방식으로 계산했는지 표시
    }

In [18]:
def train_epoch(model, dataloader, criterion, optimizer, device, epoch):
    """한 에폭 훈련"""
    model.train()
    total_loss = 0.0
    total_metrics = {"mse": 0.0, "rmse": 0.0, "mae": 0.0, "mape": 0.0, "valid_count": 0}
    
    pbar = tqdm(
        enumerate(dataloader), 
        total=len(dataloader),
        desc=f"Epoch {epoch} [Train]",
        leave=False
    )
    
    for batch_idx, batch in pbar:
        continuous_data = batch["continuous_data"].to(device)
        categorical_data = batch["categorical_data"].to(device)
        targets = batch["targets"].to(device)
        masks = batch["masks"].to(device)
        sequence_lengths = batch["sequence_lengths"]
        
        optimizer.zero_grad()
        
        # Forward pass
        predictions = model(continuous_data, categorical_data, masks, sequence_lengths)
        loss = criterion(predictions, targets, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # 메트릭 계산
        with torch.no_grad():
            batch_metrics = compute_metrics(predictions, targets, masks)
        
        total_loss += loss.item()
        for key in ["mse", "rmse", "mae", "mape"]:
            total_metrics[key] += batch_metrics[key]
        total_metrics["valid_count"] += batch_metrics["valid_count"]
        
        # 진행바 업데이트
        pbar.set_postfix({
            "Loss": f"{loss.item():.4f}",
            "MAPE": f"{batch_metrics['mape']:.2f}%"
        })
    
    pbar.close()
    
    # 평균 계산
    avg_loss = total_loss / len(dataloader)
    for key in ["mse", "rmse", "mae", "mape"]:
        total_metrics[key] = total_metrics[key] / len(dataloader)
    
    return avg_loss, total_metrics


def validate_epoch(model, dataloader, criterion, device, epoch=None):
    """검증 에폭"""
    model.eval()
    total_loss = 0.0
    total_metrics = {"mse": 0.0, "rmse": 0.0, "mae": 0.0, "mape": 0.0, "valid_count": 0}
    
    desc = f"Epoch {epoch} [Val]" if epoch is not None else "Validation"
    pbar = tqdm(dataloader, desc=desc, leave=False)
    
    with torch.no_grad():
        for batch in pbar:
            continuous_data = batch["continuous_data"].to(device)
            categorical_data = batch["categorical_data"].to(device)
            targets = batch["targets"].to(device)
            masks = batch["masks"].to(device)
            sequence_lengths = batch["sequence_lengths"]
            
            predictions = model(continuous_data, categorical_data, masks, sequence_lengths)
            loss = criterion(predictions, targets, masks)
            
            batch_metrics = compute_metrics(predictions, targets, masks)
            
            total_loss += loss.item()
            for key in ["mse", "rmse", "mae", "mape"]:
                total_metrics[key] += batch_metrics[key]
            total_metrics["valid_count"] += batch_metrics["valid_count"]
            
            pbar.set_postfix({
                "Loss": f"{loss.item():.4f}",
                "MAPE": f"{batch_metrics['mape']:.2f}%"
            })
    
    pbar.close()
    
    avg_loss = total_loss / len(dataloader)
    for key in ["mse", "rmse", "mae", "mape"]:
        total_metrics[key] = total_metrics[key] / len(dataloader)
    
    return avg_loss, total_metrics

In [19]:
def train_model(model, train_loader, val_loader, training_config, device, save_path):
    """메인 훈련 루프"""
    
    num_epochs = training_config.get("num_epochs", 100)
    learning_rate = training_config.get("learning_rate", 1e-3)
    patience = training_config.get("patience", 20)
    padding_value = training_config.get("padding_value", 0.0)
    
    # 손실 함수 및 옵티마이저
    criterion = MSELoss(padding_value)
    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=patience//2, verbose=True)
    
    model = model.to(device)
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    logger.info(f"훈련 시작: {num_epochs} 에폭, 학습률 {learning_rate}")
    
    # 에폭 진행바
    epoch_pbar = tqdm(range(1, num_epochs + 1), desc="Training Progress")
    
    for epoch in epoch_pbar:
        train_loss, train_metrics = train_epoch(
            model, train_loader, criterion, optimizer, device, epoch
        )
        val_loss, val_metrics = validate_epoch(
            model, val_loader, criterion, device, epoch
        )
        
        scheduler.step(val_loss)
        
        # 로그 출력
        logger.info(
            f"Epoch {epoch:3d}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, "
            f'Train MAPE={train_metrics["mape"]:.2f}%, Val MAPE={val_metrics["mape"]:.2f}%'
        )
        
        # 진행바 업데이트
        epoch_pbar.set_postfix({
            "T_Loss": f"{train_loss:.4f}",
            "V_Loss": f"{val_loss:.4f}",
            "V_MAPE": f'{val_metrics["mape"]:.2f}%',
            "Best": f"{best_val_loss:.4f}",
            "Patience": f"{patience_counter}/{patience}"
        })
        
        # 최고 모델 저장
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_metrics': val_metrics,
                'train_metrics': train_metrics
            }, save_path)
            
            logger.info(f"  → Best model saved! (Val Loss: {val_loss:.4f})")
        else:
            patience_counter += 1
        
        # 조기 종료
        if patience_counter >= patience:
            logger.info(f"Early stopping at epoch {epoch}")
            break
    
    epoch_pbar.close()
    
    return {
        'best_val_loss': best_val_loss
    }


def evaluate_model(
    model, 
    test_loader, 
    device, 
    model_path,
    eval_strategy: str = "target_only",  # "target_only", "average", "weighted_average", "all"
    weight_type: str = "linear",  # "linear", "exponential", "uniform"
    return_all_predictions: bool = False,
    verbose: bool = True
):
    """
    통합 모델 평가 함수
    
    Args:
        model: 평가할 모델
        test_loader: 테스트 데이터 로더
        device: 실행 디바이스
        model_path: 모델 경로
        eval_strategy: 평가 전략
            - "target_only": 각 window의 타겟 시점만 평가
            - "average": 중복 예측의 평균 사용
            - "weighted_average": 가중 평균 사용
            - "all": 모든 예측 포함 (중복 허용)
        weight_type: 가중 평균 시 가중치 타입 ("linear", "exponential", "uniform")
        return_all_predictions: 모든 중간 예측값도 반환할지 여부
        verbose: 상세 로그 출력 여부
    
    Returns:
        결과 딕셔너리
    """
    
    if verbose:
        logger.info(f"모델 로드: {model_path}")
        logger.info(f"평가 전략: {eval_strategy}")
        if eval_strategy == "weighted_average":
            logger.info(f"가중치 타입: {weight_type}")
    
    # 모델 로드
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    model = model.to(device)
    model.eval()
    
    padding_value = 0
    
    # 평가 전략에 따른 예측 수집
    if eval_strategy in ["average", "weighted_average"]:
        prediction_accumulator = defaultdict(lambda: {'predictions': [], 'targets': []})
    else:
        structured_predictions = []
        all_predictions = []
        all_targets = []
    
    # 모든 중간 예측 저장 (옵션)
    if return_all_predictions:
        all_raw_predictions = []
    
    total_loss = 0.0
    num_batches = 0
    
    if verbose:
        logger.info("테스트 시작...")
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing", disable=not verbose):
            continuous_data = batch["continuous_data"].to(device)
            categorical_data = batch["categorical_data"].to(device)
            targets = batch["targets"].to(device)
            masks = batch["masks"].to(device)
            sequence_lengths = batch.get("sequence_lengths", None)
            
            # 구조 정보 추출
            timekey_hrs = batch.get("timekey_hrs", [])
            oper_ids_list = batch.get("oper_ids_list", [])
            window_timekeys = batch.get("window_timekeys", [])
            target_groups = batch.get("target_group", [])
            
            # 예측
            predictions = model(continuous_data, categorical_data, masks, sequence_lengths)
            
            # 손실 계산 (선택적)
            if eval_strategy == "all":
                criterion = MSELoss(padding_value)
                loss = criterion(predictions, targets, masks)
                total_loss += loss.item()
                num_batches += 1
            
            # CPU로 변환
            predictions_cpu = predictions.cpu().numpy()
            targets_cpu = targets.cpu().numpy()
            masks_cpu = masks.cpu().numpy()
            
            batch_size = predictions_cpu.shape[0]
            
            for sample_idx in range(batch_size):
                sample_predictions = predictions_cpu[sample_idx]
                sample_targets = targets_cpu[sample_idx]
                sample_masks = masks_cpu[sample_idx]
                
                # 메타 정보 추출
                if sample_idx < len(oper_ids_list):
                    oper_ids = oper_ids_list[sample_idx]
                else:
                    oper_ids = [None] * len(sample_predictions)
                
                if sample_idx < len(window_timekeys):
                    window_times = window_timekeys[sample_idx]
                else:
                    window_times = [None] * 10
                
                if sample_idx < len(target_groups):
                    target_group = target_groups[sample_idx]
                else:
                    target_group = None
                
                if sample_idx < len(timekey_hrs):
                    timekey_hr = timekey_hrs[sample_idx]
                else:
                    timekey_hr = None
                
                # 전략별 처리
                if eval_strategy == "target_only":
                    _process_target_only(
                        sample_predictions, sample_targets, sample_masks,
                        window_times, target_group, timekey_hr, oper_ids,
                        structured_predictions, all_predictions, all_targets
                    )
                
                elif eval_strategy in ["average", "weighted_average"]:
                    _accumulate_predictions(
                        sample_predictions, sample_targets, sample_masks,
                        window_times, target_group, timekey_hr, oper_ids,
                        prediction_accumulator
                    )
                
                elif eval_strategy == "all":
                    _process_all_predictions(
                        sample_predictions, sample_targets, sample_masks,
                        window_times, target_group, timekey_hr, oper_ids,
                        structured_predictions, all_predictions, all_targets
                    )
                
                # 모든 예측 저장 (옵션)
                if return_all_predictions:
                    for i in range(len(sample_predictions)):
                        if sample_masks[i]:
                            all_raw_predictions.append({
                                'prediction': sample_predictions[i],
                                'target': sample_targets[i],
                                'timekey': window_times[i] if i < len(window_times) else timekey_hr,
                                'oper_id': oper_ids[i] if i < len(oper_ids) else None
                            })
    
    # 평균 계산 (average/weighted_average 전략)
    if eval_strategy in ["average", "weighted_average"]:
        structured_predictions, all_predictions, all_targets = _compute_averages(
            prediction_accumulator, 
            eval_strategy, 
            weight_type
        )
    
    # 메트릭 계산
    metrics = _compute_metrics(all_predictions, all_targets)
    
    if verbose:
        logger.info(f"테스트 결과: RMSE={metrics['rmse']:.4f}, "
                   f"MAE={metrics['mae']:.4f}, MAPE={metrics['mape']:.2f}%")
        logger.info(f"평가된 데이터 포인트: {len(all_predictions):,}개")
    
    result = {
        "metrics": metrics,
        "predictions": all_predictions,
        "targets": all_targets,
        "structured_predictions": structured_predictions,
        "eval_strategy": eval_strategy
    }
    
    if eval_strategy == "all" and num_batches > 0:
        result["avg_loss"] = total_loss / num_batches
    
    if return_all_predictions:
        result["all_raw_predictions"] = all_raw_predictions
    
    return result


def _process_target_only(
    predictions, targets, masks, window_times, target_group, 
    timekey_hr, oper_ids, structured_predictions, all_predictions, all_targets
):
    """타겟 시점만 처리"""
    # Window의 마지막 시점 (타겟)
    window_size = min(10, len(masks))
    valid_window_size = sum(masks[:window_size])
    
    if valid_window_size > 0:
        target_idx = valid_window_size - 1
        if masks[target_idx]:
            structured_predictions.append({
                'timekey_hr': window_times[target_idx] if window_times[target_idx] else timekey_hr,
                'oper_id': f"group_{target_group}" if target_group else "unknown",
                'predicted': float(predictions[target_idx]),
                'actual': float(targets[target_idx])
            })
            all_predictions.append(float(predictions[target_idx]))
            all_targets.append(float(targets[target_idx]))
    
    # Group 부분 (개별 작업자)
    group_start = 10
    for seq_idx in range(group_start, len(predictions)):
        if masks[seq_idx] and seq_idx < len(oper_ids) and oper_ids[seq_idx] is not None:
            structured_predictions.append({
                'timekey_hr': timekey_hr,
                'oper_id': oper_ids[seq_idx],
                'predicted': float(predictions[seq_idx]),
                'actual': float(targets[seq_idx])
            })
            all_predictions.append(float(predictions[seq_idx]))
            all_targets.append(float(targets[seq_idx]))


def _accumulate_predictions(
    predictions, targets, masks, window_times, target_group,
    timekey_hr, oper_ids, accumulator
):
    """예측 누적 (평균 계산용)"""
    # Window 부분
    for seq_idx in range(min(10, len(window_times))):
        if masks[seq_idx] and window_times[seq_idx] is not None:
            key = (window_times[seq_idx], f"group_{target_group}")
            accumulator[key]['predictions'].append(float(predictions[seq_idx]))
            accumulator[key]['targets'].append(float(targets[seq_idx]))
    
    # Group 부분
    group_start = 10
    for seq_idx in range(group_start, len(predictions)):
        if masks[seq_idx] and seq_idx < len(oper_ids) and oper_ids[seq_idx] is not None:
            key = (timekey_hr, oper_ids[seq_idx])
            accumulator[key]['predictions'].append(float(predictions[seq_idx]))
            accumulator[key]['targets'].append(float(targets[seq_idx]))


def _process_all_predictions(
    predictions, targets, masks, window_times, target_group,
    timekey_hr, oper_ids, structured_predictions, all_predictions, all_targets
):
    """모든 예측 처리 (중복 포함)"""
    for seq_idx in range(len(predictions)):
        if masks[seq_idx]:
            if seq_idx < 10:  # Window 부분
                timekey = window_times[seq_idx] if seq_idx < len(window_times) else timekey_hr
                oper_id = f"group_{target_group}" if target_group else "unknown"
            else:  # Group 부분
                timekey = timekey_hr
                oper_id = oper_ids[seq_idx] if seq_idx < len(oper_ids) else "unknown"
            
            if timekey is not None:
                structured_predictions.append({
                    'timekey_hr': timekey,
                    'oper_id': oper_id,
                    'predicted': float(predictions[seq_idx]),
                    'actual': float(targets[seq_idx])
                })
                all_predictions.append(float(predictions[seq_idx]))
                all_targets.append(float(targets[seq_idx]))


def _compute_averages(accumulator, strategy, weight_type):
    """누적된 예측의 평균 계산"""
    structured_predictions = []
    all_predictions = []
    all_targets = []
    
    for (timekey, oper_id), values in accumulator.items():
        preds = values['predictions']
        targs = values['targets']
        
        if len(preds) == 0:
            continue
        
        # 타겟은 첫 번째 값 사용 (모두 같아야 함)
        target = targs[0]
        
        # 예측 평균 계산
        if strategy == "average":
            avg_pred = np.mean(preds)
        elif strategy == "weighted_average":
            avg_pred = _weighted_average(preds, weight_type)
        else:
            avg_pred = np.mean(preds)
        
        structured_predictions.append({
            'timekey_hr': timekey,
            'oper_id': oper_id,
            'predicted': avg_pred,
            'actual': target,
            'num_predictions': len(preds)
        })
        
        all_predictions.append(avg_pred)
        all_targets.append(target)
    
    return structured_predictions, all_predictions, all_targets


def _weighted_average(predictions, weight_type="linear"):
    """가중 평균 계산"""
    n = len(predictions)
    if n == 1:
        return predictions[0]
    
    if weight_type == "linear":
        # 최신일수록 높은 가중치
        weights = np.arange(1, n + 1)
    elif weight_type == "exponential":
        # 지수적으로 증가하는 가중치
        weights = np.exp(np.arange(n))
    else:  # uniform
        weights = np.ones(n)
    
    weights = weights / weights.sum()
    return np.average(predictions, weights=weights)


def _compute_metrics(predictions, targets, exclude_zeros=True):
    """
    평가 메트릭 계산
    
    Args:
        predictions: 예측값 배열
        targets: 실제값 배열
        exclude_zeros: True면 MAPE/SMAPE 계산 시 타겟 0 제외, 
                      False면 0도 포함 (epsilon으로 처리)
    """
    predictions = np.array(predictions)
    targets = np.array(targets)
    
    # 기본 메트릭 (모든 데이터 포함)
    mse = np.mean((predictions - targets) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(predictions - targets))
    
    if exclude_zeros:
        # 타겟이 0이 아닌 경우만 필터링
        non_zero_mask = targets != 0
        non_zero_count = non_zero_mask.sum()
        
        if non_zero_count > 0:
            filtered_predictions = predictions[non_zero_mask]
            filtered_targets = targets[non_zero_mask]
            
            abs_errors = np.abs(filtered_predictions - filtered_targets)
            abs_targets = np.abs(filtered_targets)
            
            # MAPE
            mape = np.mean(abs_errors / abs_targets * 100)
            
            # SMAPE  
            smape = 100 * np.mean(
                2 * abs_errors / (abs_targets + np.abs(filtered_predictions) + 1e-8)
            )
        else:
            mape = np.nan
            smape = np.nan
            
    else:
        # 0도 포함하여 계산 (epsilon 사용)
        epsilon = 1e-8
        abs_targets = np.abs(targets)
        abs_errors = np.abs(predictions - targets)
        
        # MAPE - epsilon으로 0 처리
        safe_targets = np.maximum(abs_targets, epsilon)
        mape = np.mean(abs_errors / safe_targets * 100)
        
        # SMAPE - epsilon으로 0 처리
        denominator = (abs_targets + np.abs(predictions)) / 2 + epsilon
        smape = 100 * np.mean(abs_errors / denominator)
        
        non_zero_count = (targets != 0).sum()
    
    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "mape": mape,
        "smape": smape,
        "valid_count": len(predictions),
        "non_zero_count": non_zero_count,
        "zero_ratio": (len(targets) - non_zero_count) / len(targets) * 100,
        "exclude_zeros": exclude_zeros  # 어떤 방식으로 계산했는지 표시
    }


## 🎯 메인 실행

In [20]:
parser = argparse.ArgumentParser(description="시계열 시퀀스 모델링")
parser.add_argument("--config-dir", default="configs", help="설정 파일 디렉토리")
parser.add_argument("--mode", choices=["train", "eval"], default="train", help="실행 모드")
parser.add_argument("--model-path", default=None, help="평가용 모델 경로")
parser.add_argument("--gpu", type=int, default=0, help="GPU 번호")
parser.add_argument("--exp-name", default=None, help="실험명")

args = parser.parse_args([])

# 설정 로드
config = load_config(args.config_dir)
set_random_seeds(42)

# 실험명 설정
if args.exp_name:
    exp_name = args.exp_name
else:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_type = config.get("model_type", "lstm")
    exp_name = f"{model_type}_{timestamp}"

# 저장 디렉토리
save_dir = config.get("save_dir", "models")
os.makedirs(save_dir, exist_ok=True)
model_save_path = os.path.join(save_dir, f"{exp_name}.pth")

# 디바이스 설정
device = torch.device(f"cuda:{args.gpu}" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

# 데이터로더 생성
logger.info("데이터 로딩 중...")
train_loader, val_loader, test_loader, categorical_processor = create_dataloaders(config)

# 모델 생성
vocab_sizes = categorical_processor.get_vocab_sizes()
continuous_dim = len(config["continuous_columns"])

model = create_model(config, vocab_sizes, continuous_dim)

if args.mode == "train":
    # 훈련
    logger.info("훈련 시작...")
    train_results = train_model(
        model, train_loader, val_loader, config, device, model_save_path
    )
    
    logger.info("훈련 완료, 테스트 시작...")
    test_results = evaluate_model(model, test_loader, device, model_save_path)
    
else:
    # 평가
    if not args.model_path:
        raise ValueError("--model-path must be provided in eval mode")
    test_results = evaluate_model(model, test_loader, device, args.model_path)

# 결과 저장
results = {
    "exp_name": exp_name,
    "config": config,
    "test_metrics": test_results["metrics"],
    "model_info": {
        "total_parameters": sum(p.numel() for p in model.parameters()),
        "model_type": config.get("model_type", "lstm")
    }
}

results_path = os.path.join(save_dir, f"{exp_name}_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2, default=str)

# 예측 결과 저장
if "structured_predictions" in test_results and test_results["structured_predictions"]:
    # 구조화된 예측 결과 저장 (timekey_hr, oper_id 포함)
    structured_df = pd.DataFrame(test_results["structured_predictions"])
    structured_df["error"] = structured_df["predicted"] - structured_df["actual"]
    structured_df["abs_error"] = structured_df["error"].abs()
    structured_df["abs_percent_error"] = (
        structured_df["abs_error"] / structured_df["actual"].abs().clip(lower=1e-8) * 100
    )
    
    # 구조화된 결과를 메인 예측 파일로 저장
    predictions_path = os.path.join(save_dir, f"{exp_name}_predictions.csv")
    structured_df.to_csv(predictions_path, index=False)
    
    logger.info(f"  - 구조화된 예측 결과: {predictions_path}")
    logger.info(f"  - 저장된 예측 개수: {len(structured_df):,}개")
    logger.info(f"  - 고유한 timekey_hr: {structured_df['timekey_hr'].nunique()}개")
    logger.info(f"  - 고유한 oper_id: {structured_df['oper_id'].nunique()}개")
    
else:
    # 구조화된 정보가 없는 경우 기본 방식으로 저장 (호환성 유지)
    predictions_df = pd.DataFrame({
        "actual": test_results["targets"],
        "predicted": test_results["predictions"],
        "residual": test_results["targets"] - test_results["predictions"],
        "abs_error": np.abs(test_results["targets"] - test_results["predictions"]),
        "abs_percent_error": (
            np.abs(test_results["targets"] - test_results["predictions"]) / 
            np.maximum(np.abs(test_results["targets"]), 1e-8) * 100
        )
    })
    
    predictions_path = os.path.join(save_dir, f"{exp_name}_predictions.csv")
    predictions_df.to_csv(predictions_path, index=False)
    
    logger.info(f"  - 기본 예측 결과: {predictions_path}")
    logger.info(f"  - 저장된 예측 개수: {len(predictions_df):,}개")


2025-09-05 22:39:35,553 - INFO - Using device: cuda:0
2025-09-05 22:39:35,554 - INFO - 데이터 로딩 중...
2025-09-05 22:39:36,106 - INFO - 범주형 변수별 고유값 개수:
2025-09-05 22:39:36,107 - INFO -   oper_group: 2개
2025-09-05 22:39:36,108 - INFO -   days: 7개
2025-09-05 22:39:36,109 - INFO -   shift: 3개
2025-09-05 22:39:36,109 - INFO -   x1: 2개
2025-09-05 22:39:38,754 - INFO - Window-Group 데이터셋 구성 완료:
2025-09-05 22:39:38,756 - INFO -   - 총 시퀀스 수: 141
2025-09-05 22:39:38,757 - INFO -   - Window 크기: 10
2025-09-05 22:39:38,757 - INFO -   - Stride: 1
2025-09-05 22:39:38,759 - INFO -   - 최대 그룹 크기: 1
2025-09-05 22:39:38,760 - INFO -   - 전체 시퀀스 길이: 11
2025-09-05 22:39:38,761 - INFO -   - 특성 차원: 24
2025-09-05 22:39:38,762 - INFO -   - 패딩값: 0.0
2025-09-05 22:39:38,978 - INFO - Window-Group 데이터셋 구성 완료:
2025-09-05 22:39:38,979 - INFO -   - 총 시퀀스 수: 10
2025-09-05 22:39:38,980 - INFO -   - Window 크기: 10
2025-09-05 22:39:38,981 - INFO -   - Stride: 1
2025-09-05 22:39:38,982 - INFO -   - 최대 그룹 크기: 1
2025-09-05 22:39:3